# Solutions · Chapter 03-04 · Probability, by counting

E6, E10 and E16 are the ones to attempt first. E6's answer is much larger than most people guess, and
E10 is the calculation that most often changes how someone reads a medical letter.

In [ ]:
import numpy as np
import pandas as pd

counts = pd.DataFrame({"alarm": [36, 96], "no alarm": [4, 864]}, index=["cracked", "sound"])
counts.index.name = "reality"
print(counts.to_string())

## E1 · Two counts, two denominators

**`P(alarm | cracked)`** counts the 36 alarmed-and-cracked bikes out of **the 40 cracked bikes**. The
denominator is every bike that really has a crack.

**`P(cracked | alarm)`** counts the same 36 out of **the 132 bikes that alarmed**. The denominator is
every bike the sensor flagged.

## E2 · Why a good detector mostly cries wolf

Because the group it can be wrong about is enormous compared with the group it can be right about. A
1% false-alarm rate applied to 99,000 healthy cases produces 990 false alarms; a 99% hit rate applied
to 1,000 real cases produces 990 true ones. The errors are drawn from a much larger pool, so they
keep pace with the successes however good the detector is. Making the detector better shrinks its
error *rate*; it does not shrink the *population* that rate is applied to.

## E3 · Independence in counts, without the word probability

**Every cell equals its row total times its column total, divided by the grand total.** Equivalently:
the split of one classification is the same in every level of the other - the red bikes break in the
same proportion as the black ones, and as the fleet.

## E4 · Three probabilities from the no-alarm side

In [ ]:
print("P(sound | no alarm) = 864 / 868  = %.4f" % (864 / 868))
print("P(no alarm | sound) = 864 / 960  = %.4f" % (864 / 960))
print("P(sound)            = 960 / 1000 = %.4f" % (960 / 1000))

**`P(no alarm | sound)` = 0.9000 and `P(sound)` = 0.9600 are the two that are close** - because the
fleet is mostly sound, so conditioning on soundness barely changes anything and the sensor's 10%
false-alarm rate accounts for the rest of the gap.

**`P(sound | no alarm)` = 0.9954 is the different one**, and it is different for a good reason: a
silent sensor is genuinely strong evidence. Only 4 of the 868 silent bikes are cracked. That is the
mirror image of the chapter's result - **the same sensor whose alarms mean little has silences that
mean a great deal**, because there are so few cracks for it to miss.

Worth carrying: a test can be far more informative in one direction than the other. For rare
conditions, a negative result is usually the trustworthy one.

## E5 · The same sensor, a different workshop

In [ ]:
def build_table(fleet, cracked, hit_rate=0.90, false_alarm_rate=0.10):
    sound = fleet - cracked
    true_alarms = cracked * hit_rate
    false_alarms = sound * false_alarm_rate
    return pd.DataFrame(
        {"alarm": [true_alarms, false_alarms],
         "no alarm": [cracked - true_alarms, sound - false_alarms]},
        index=["cracked", "sound"],
    )


second = build_table(500, 100)
print(second.to_string())
print()
print("P(cracked | alarm) = %.0f / %.0f = %.4f"
      % (second.loc["cracked", "alarm"], second["alarm"].sum(),
         second.loc["cracked", "alarm"] / second["alarm"].sum()))
print("the first workshop got 0.2727")

**0.6923 against 0.2727**, from an identical sensor.

**The one sentence:** the second workshop's bikes are cracked 20% of the time instead of 4%, so there
are five times as many real cracks to find and a quarter as many sound frames to raise false alarms
about.

Everything about the equipment is unchanged. This is the reason a vendor's accuracy figure is not
transferable between sites, and the reason a model's precision reported on one population tells you
almost nothing about its precision on another.

## E6 · Three cables from the same batch

In [ ]:
bad_batch_rate, fail_if_bad, fail_if_good = 0.02, 0.45, 0.001

one = bad_batch_rate * fail_if_bad + (1 - bad_batch_rate) * fail_if_good
rows = []
for k in [2, 3, 4]:
    actual = bad_batch_rate * fail_if_bad ** k + (1 - bad_batch_rate) * fail_if_good ** k
    rows.append({"cables that must all fail": k,
                 "if independent": "%.3e" % one ** k,
                 "actual": "%.3e" % actual,
                 "ratio": round(actual / one ** k, 1)})
print(pd.DataFrame(rows).to_string(index=False))

**Three cables: 1,833 times the independent estimate**, against 40.7 for two.

The ratio grows roughly by a factor of `fail_if_bad / one` with each extra component - because the
independent calculation raises a small number to a higher power while the real one is dominated by
the bad-batch term, which barely shrinks.

**The practical consequence is the opposite of the intuition.** Adding redundancy looks like it
multiplies safety: two cables should be a hundred times safer than one, three a hundred times safer
again. When the components share a cause, redundancy buys far less than the arithmetic promises, and
**the more redundancy you add, the larger the gap between the promise and the reality**.

This is why serious reliability engineering insists on *diverse* redundancy - different suppliers,
different designs, different power sources - rather than more copies of the same thing. Identical
copies fail together.

## E7 · `table_probabilities`

In [ ]:
def table_probabilities(table):
    n = table.values.sum()
    rows, cols = list(table.index), list(table.columns)

    print("MARGINALS")
    for r in rows:
        print("  P(%-8s) = %5.0f / %d = %.4f" % (r, table.loc[r].sum(), n, table.loc[r].sum() / n))
    for c in cols:
        print("  P(%-8s) = %5.0f / %d = %.4f" % (c, table[c].sum(), n, table[c].sum() / n))

    print("JOINT")
    for r in rows:
        for c in cols:
            print("  P(%-8s and %-8s) = %5.0f / %d = %.4f"
                  % (r, c, table.loc[r, c], n, table.loc[r, c] / n))

    print("CONDITIONAL")
    for r in rows:
        for c in cols:
            print("  P(%-8s | %-8s) = %5.0f / %5.0f = %.4f    "
                  "P(%-8s | %-8s) = %5.0f / %5.0f = %.4f"
                  % (r, c, table.loc[r, c], table[c].sum(), table.loc[r, c] / table[c].sum(),
                     c, r, table.loc[r, c], table.loc[r].sum(), table.loc[r, c] / table.loc[r].sum()))


table_probabilities(counts)

The conditional block prints both directions side by side on purpose. `P(cracked | alarm)` = 0.2727
and `P(alarm | cracked)` = 0.9000 appear on the same line, from the same cell, which makes the point
harder to forget than any amount of prose.

## E8 · `independence_gap`

In [ ]:
colour = pd.DataFrame({"cracked": [16, 24], "sound": [384, 576]}, index=["red", "black"])
colour.index.name = "frame colour"


def independence_gap(table):
    n = table.values.sum()
    expected = pd.DataFrame(
        table.sum(axis=1).to_numpy()[:, None] * table.sum(axis=0).to_numpy()[None, :] / n,
        index=table.index, columns=table.columns)
    difference = table - expected
    summary = float(((difference ** 2) / expected).values.sum())     # the chi-squared statistic
    return difference, summary


for name, table in [("COLOUR", colour), ("SENSOR", counts)]:
    difference, summary = independence_gap(table)
    print(name)
    print(difference.round(2).to_string())
    print("  summary of the discrepancy: %.2f\n" % summary)

Colour: every difference is **exactly zero**, and the summary is 0.00 - independence, with nothing
left over.

Sensor: cracked-and-alarmed is **30.7 above** what independence predicts, and the summary is 214.50.

The summary used here is the chi-squared statistic, `sum((observed - expected)^2 / expected)`. It is
worth seeing it constructed rather than invoked: it is nothing more than **how far the table is from
what independence would produce, with each cell's miss scaled by how many were expected there**. A
statistical test turns that number into a probability; the number itself is already informative.

## E9 · Independent overall, dependent within groups

In [ ]:
# Two workshops. Within each, the sensor works. Pooled, the association vanishes.
north = pd.DataFrame({"alarm": [36, 24], "no alarm": [24, 36]}, index=["cracked", "sound"])
south = pd.DataFrame({"alarm": [24, 36], "no alarm": [36, 24]}, index=["cracked", "sound"])
pooled = north + south

for name, table in [("NORTH", north), ("SOUTH", south), ("POOLED", pooled)]:
    n = table.values.sum()
    p_alarm_given_cracked = table.loc["cracked", "alarm"] / table.loc["cracked"].sum()
    p_alarm = table["alarm"].sum() / n
    print("%-7s P(alarm | cracked) = %.3f   P(alarm) = %.3f   %s"
          % (name, p_alarm_given_cracked, p_alarm,
             "independent" if abs(p_alarm_given_cracked - p_alarm) < 1e-9 else "DEPENDENT"))

In **north**, knowing a bike is cracked raises the chance of an alarm from 0.500 to 0.600. In
**south** it *lowers* it, from 0.500 to 0.400. Pooled, the two effects cancel exactly and the table
is perfectly independent - 0.500 either way.

**Independence is not a property that survives pooling or splitting.** It is 02-06's Simpson's
paradox in probability language, and it carries the same warning: a variable can look irrelevant
overall while mattering, in opposite directions, inside subgroups. Any check for independence is a
check on the table you happen to have drawn.

## E10 · The screening programme

In [ ]:
population, prevalence = 100_000, 1 / 2000
ill = population * prevalence
well = population - ill
true_positives = ill * 0.99
false_positives = well * 0.01

screening = pd.DataFrame(
    {"test positive": [true_positives, false_positives],
     "test negative": [ill - true_positives, well - false_positives]},
    index=["has the condition", "does not"])
print(screening.round(1).to_string())
print()
print("P(has it | tested positive) = %.1f / %.1f = %.4f  -> about 1 in %.0f"
      % (true_positives, true_positives + false_positives,
         true_positives / (true_positives + false_positives),
         (true_positives + false_positives) / true_positives))

**4.7% - about 1 in 21.** Of 1,049 positive results, 999 are wrong.

**The sentence for the letter:**

> "Your screening test was positive. This test is designed to be very sensitive, so that it almost
> never misses the condition - which means it also flags many people who do not have it. Because the
> condition is rare, about 1 in 21 people with a positive result turn out to have it, and the other
> 20 do not. The next step is a confirmatory test, which is more specific, and most people who reach
> this stage are found to be clear."

Three properties that letter has and most do not: it gives the number, it explains *why* the number
is low without implying the test failed, and it says what happens next. The commonest real-world
failure is sending a bare "positive" result, which the recipient reads as 99%.

## E11 · The fraud model

In [ ]:
transactions = 100_000
fraud = 0.003 * transactions
flagged = 0.02 * transactions
caught = 0.80 * fraud

print("transactions %d   fraudulent %.0f   flagged %.0f   fraud caught %.0f"
      % (transactions, fraud, flagged, caught))
print()
print("recall    = %.0f / %.0f = %.3f" % (caught, fraud, caught / fraud))
print("precision = %.0f / %.0f = %.3f" % (caught, flagged, caught / flagged))
print("false alarms sent to customers: %.0f" % (flagged - caught))

**Recall 0.800, precision 0.120.** Of 2,000 flagged transactions, 1,760 are legitimate.

**What raising recall to 95% does to precision: it lowers it, and by more than the recall gain.**

The reason needs no arithmetic. To catch the last 15% of fraud - the 45 hardest cases, the ones that
look most like ordinary transactions - the model has to accept transactions that look ordinary. But
almost all ordinary-looking transactions *are* ordinary: there are 99,700 legitimate ones and 300
frauds. Every relaxation of the threshold admits a few more frauds and a great many more legitimate
customers, because that is the ratio available in the pool.

**This trade never goes away.** Precision and recall move against each other whenever a threshold is
moved, and the exchange rate is set by the base rate - which is why module 07 plots the whole curve
instead of quoting one point, and why "improve recall" is not a complete instruction without saying
what precision is acceptable.

## E12 · "Three data centres, so one in a billion"

**The assumption: that the three outages are independent** - that knowing one data centre is down
tells you nothing about the others.

**Two concrete reasons it fails here:**

1. **Shared dependencies.** A common DNS provider, certificate authority, deployment pipeline, or
   configuration push. A bad deploy reaches all three within minutes, and the correlation is near
   total.
2. **Correlated demand.** The event that takes down the first data centre - a traffic spike, a
   cascading retry storm - is often the same event that then hits the others, and the failover load
   from the first makes the second more likely to fail, not less.

Regional power and network events belong on the list too, unless the sites are genuinely far apart.

**Direction: the estimate is far too optimistic**, exactly like the cable calculation. And the error
is worse than 40x here, because the shared causes in infrastructure are stronger than a batch effect:
the realistic figure is dominated by "how often does something take out all three at once", which is
much closer to the single-site failure rate than to its cube.

The honest version: "each site has 99.9% uptime individually; our simultaneous-failure risk is
dominated by shared dependencies, which we estimate separately."

## E13 · Precision and recall for a product manager

> "Think about the bike sensor. Of the forty bikes that really have a cracked frame, it catches
> thirty-six - that is recall, and it answers 'how much of the real problem do we find'. But it also
> flags ninety-six perfectly sound bikes, so of the hundred and thirty-two alarms only thirty-six are
> real - that is precision, and it answers 'when we flag something, how often are we right'. They are
> the same thirty-six bikes divided by two different totals, which is why you cannot summarise the
> system with one number. I would optimise recall when missing a case is much more expensive than a
> false alarm - a cracked frame that injures someone, fraud, a safety fault. I would optimise
> precision when acting on a flag is expensive or annoying - blocking a customer's card, sending a
> mechanic out, interrupting a user - because there each false alarm has a real cost and most flags
> would be false."

Five sentences, both metrics grounded in one table of counts, and a decision rule that a product
manager can apply without doing arithmetic.

## E14 · The email filter

In [ ]:
daily, spam_share, marked_share, correct_when_marked = 200, 0.06, 0.04, 0.95

genuine_spam = daily * spam_share
marked = daily * marked_share
correctly_marked = marked * correct_when_marked

print("messages a day            : %d" % daily)
print("genuinely spam            : %.0f" % genuine_spam)
print("marked as spam            : %.0f" % marked)
print("of those, actually spam   : %.1f" % correctly_marked)
print()
print("legitimate mail sent to the spam folder : %.1f per day" % (marked - correctly_marked))
print("spam that reaches the inbox             : %.1f per day" % (genuine_spam - correctly_marked))

**0.4 legitimate emails lost per day - roughly one every two and a half days - and 4.4 spam messages
reaching the inbox.**

**Which error is worse: losing legitimate mail, by a wide margin.** Deleting spam from the inbox
costs two seconds and the user knows it happened. A legitimate message in the spam folder is invisible
- the user does not know it exists, does not know to look, and the cost lands somewhere else entirely
if it was a job offer, an invoice, or a message from a doctor.

**What that implies for the threshold: set it conservatively**, marking only what the filter is very
confident about. That means accepting more spam in the inbox, which is the cheap error. The
asymmetry is severe - most people would trade fifty extra spam messages to avoid losing one real one -
and it should be reflected in the threshold rather than in a metric that treats the two errors
equally.

This is the first appearance of a theme that runs through module 07: **accuracy weights both errors
the same, and almost no real problem does.**

## E15 · For your friend

> "That 99% is the test's ability to spot the condition in people who have it - and it is genuinely
> good at that. It is not the chance that you have it. The condition is rare, so out of a hundred
> thousand people tested, about fifty actually have it and are nearly all found, while about a
> thousand healthy people get a positive result too - not because the test is bad, but because there
> are so many more healthy people for a small error rate to act on. That is why a first positive is a
> reason for the second test rather than a diagnosis, and why most people at your stage turn out to be
> clear. The number worth holding onto until then is closer to 1 in 20 than to 99 in 100."

118 words, does not undermine the test, gives them the correct number, and tells them what happens
next - which is the thing they actually need.

## E16 · Detecting the dependence from failures alone

In [ ]:
detect_rng = np.random.default_rng(3)

rows = []
for fleet in [200, 1_000, 10_000, 100_000]:
    from_bad_batch = detect_rng.random(fleet) < bad_batch_rate
    chance = np.where(from_bad_batch, fail_if_bad, fail_if_good)
    first = detect_rng.random(fleet) < chance
    second = detect_rng.random(fleet) < chance
    observed_both = int((first & second).sum())
    expected_both = first.sum() * second.sum() / fleet
    rows.append({"fleet": fleet,
                 "cables failed": int(first.sum() + second.sum()),
                 "both failed, observed": observed_both,
                 "both failed, expected if independent": round(float(expected_both), 2),
                 "ratio": round(observed_both / expected_both, 1) if expected_both > 0 else None})
print(pd.DataFrame(rows).to_string(index=False))

**Yes, and much sooner than expected.**

**The statistic:** count the bikes with *both* cables failed, and compare it with
`(first failures) x (second failures) / fleet` - the count independence predicts. That is exactly the
expected-count calculation from E8, applied to one cell.

**How large a fleet is needed:** even at **1,000 bikes** the ratio is unmistakable - eight double
failures observed against 0.23 expected. At 10,000 it is 41 against 1.04, and at 200 bikes there is
already one double failure where independence expects 0.02. The dependence is easy to
see because it multiplies the rare cell by forty, and rare cells are where dependence shows up most
strongly.

**The catch, and the reason this exercise is here.** Detecting the dependence requires *thinking to
look*. Nobody stumbles onto this: the individual failure rates are exactly as specified, every
component passes its own test, and the only visible symptom is in a joint count that a component-level
report never computes. The batch variable is not in the data at all - it was never recorded - and yet
its effect is measurable from failures alone, if you ask.

That is the general lesson of the chapter. **Independence is an empirical claim, and it is usually
checkable in data you already have.** The reason it goes unchecked is not difficulty; it is that
multiplying feels like arithmetic rather than like an assumption.

## Where to go next

**03-05 · Bayes' rule you can do on paper.** Everything in this chapter was a count divided by a
count, which works when you have the table. Bayes' rule handles the far commoner situation where you
have *rates* - a base rate and a detector's error rates - and need `P(cause | evidence)` without ever
seeing a table. It is this chapter's arithmetic, rearranged.